<a href="https://colab.research.google.com/github/momo25bend/streamsentinel/blob/main/J5_contexte_risque.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

import json

with open('/content/drive/MyDrive/streamsentinel/resultats/resultats_j4.json') as f:
    j4 = json.load(f)

print(len(j4), "éléments chargés")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
19 éléments chargés


In [12]:
import requests

def meteo_contexte(lat, lon):
    """Pluie et température : 3 jours passés + 3 jours de prévision."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "precipitation_sum,temperature_2m_max",
        "past_days": 3,
        "forecast_days": 4,   # aujourd'hui + 3 jours
        "timezone": "Europe/Paris",
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        d = r.json()["daily"]
    except Exception as e:
        return {"disponible": False, "erreur": str(e)}

    pluie = [p or 0 for p in d["precipitation_sum"]]
    temp = [t for t in d["temperature_2m_max"] if t is not None]
    # indices : 0-2 = 3 jours passés, 3 = aujourd'hui, 4-6 = 3 jours à venir
    return {
        "disponible": True,
        "pluie_72h_mm": round(sum(pluie[0:3]), 1),
        "pluie_aujourdhui_mm": round(pluie[3], 1),
        "pluie_prevue_3j_mm": round(sum(pluie[4:7]), 1),
        "temp_max_aujourdhui": d["temperature_2m_max"][3],
        "temp_max_3j": max(temp[3:7]) if temp else None,
    }

# Test sur un point de la Seine à Paris
print(meteo_contexte(48.8566, 2.3522))

{'disponible': True, 'pluie_72h_mm': 0, 'pluie_aujourdhui_mm': 0, 'pluie_prevue_3j_mm': 0, 'temp_max_aujourdhui': 24.5, 'temp_max_3j': 25.5}


In [13]:
def val(fiche, champ):
    """Valeur d'un champ du formulaire, en liste (gère simple et multiple)."""
    v = fiche["formulaire"].get(champ, {}).get("valeur")
    return v if isinstance(v, list) else [v]

def niveau(score):
    return "ELEVE" if score >= 4 else "MODERE" if score >= 2 else "FAIBLE"

CONSIGNES = {
    "humains": {
        "FAIBLE": "Pas de danger visible. Évitez tout de même de boire l'eau.",
        "MODERE": "Évitez le contact avec l'eau et lavez-vous les mains après la visite.",
        "ELEVE":  "Ne touchez pas l'eau. Signalement transmis au gestionnaire pour vérification.",
    },
    "animaux": {
        "FAIBLE": "Pas de danger visible pour les animaux.",
        "MODERE": "Empêchez votre chien de boire ou de se baigner.",
        "ELEVE":  "Tenez les animaux en laisse, loin de l'eau. Signalement transmis au gestionnaire.",
    },
}

def evaluer_risque(fiche, meteo):
    h, a, raisons = 0, 0, []   # score humains, score animaux
    aspect = val(fiche, "water_aspect")
    chaud = meteo.get("disponible") and (meteo.get("temp_max_aujourdhui") or 0) >= 25
    pluie = meteo.get("pluie_72h_mm", 0) if meteo.get("disponible") else 0

    # --- Signaux de l'image (J1 à J4) ---
    if "FOAM" in aspect:
        h += 2; a += 2; raisons.append("mousse sur l'eau : pollution possible")
    if "ALTERED_COLOR" in aspect:
        h += 2; a += 2; raisons.append("couleur de l'eau anormale")
        if chaud:
            h += 1; a += 2; raisons.append("eau colorée + chaleur : cyanobactéries possibles, dangereuses pour les chiens")
    if "MUDDY" in aspect:
        h += 1; a += 1; raisons.append("eau trouble")
    if "YES" in val(fiche, "sewage_discharge"):
        h += 3; a += 3; raisons.append("rejet d'eaux usées visible")
    if "YES" in val(fiche, "draining_pipes"):
        h += 1; a += 1; raisons.append("canalisation de drainage visible")

    sig = fiche.get("signalements", {})
    if sig.get("animal_mort", {}).get("valeur") == "YES":
        h += 3; a += 3; raisons.append("animal mort signalé (à confirmer)")
    if sig.get("dechets"):
        h += 1; a += 1; raisons.append(f"{len(sig['dechets'])} déchet(s) détecté(s) : blessure ou ingestion")

    # --- Contexte météo ---
    if pluie >= 20:
        h += 2; a += 2; raisons.append(f"{pluie} mm de pluie en 72 h : ruissellement et débordements d'égouts probables")
    elif pluie >= 10:
        h += 1; a += 1; raisons.append(f"{pluie} mm de pluie en 72 h")

    flow = str(val(fiche, "water_flow")[0])
    if flow.startswith("FAS") and pluie >= 10:
        h += 1; raisons.append("courant rapide après la pluie : risque de chute ou d'emport")
    if flow == "STAGNANT" and chaud:
        a += 1; raisons.append("eau stagnante et chaude")

    nh, na = niveau(h), niveau(a)

    # --- Tendance à 3 jours (optionnelle) ---
    tendance = "inconnue"
    if meteo.get("disponible"):
        tendance = "en hausse" if meteo["pluie_prevue_3j_mm"] >= 20 else "stable"

    return {
        "risque_humains": {"niveau": nh, "score": h, "consigne": CONSIGNES["humains"][nh]},
        "risque_animaux": {"niveau": na, "score": a, "consigne": CONSIGNES["animaux"][na]},
        "raisons": raisons or ["aucun signal de risque détecté"],
        "tendance_3j": tendance,
        "meteo": meteo,
        "a_valider_gestionnaire": "ELEVE" in (nh, na),  # jamais d'alerte directe
    }

In [14]:
from datetime import date, timedelta
from math import radians, sin, cos, asin, sqrt

BASE = "https://hubeau.eaufrance.fr/api"

def hubeau(chemin, params):
    """Essaie l'API v2 puis v1 ; renvoie la liste 'data' ou None."""
    for v in ("v2", "v1"):
        try:
            r = requests.get(f"{BASE}/{v}/hydrometrie/{chemin}", params=params, timeout=20)
            if r.status_code in (200, 206):
                return r.json().get("data", [])
        except Exception:
            pass
    return None

def distance_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return 6371 * 2 * asin(sqrt(a))

def hubeau_contexte(lat, lon, rayon_km=10):
    stations = hubeau("referentiel/stations",
                      {"latitude": lat, "longitude": lon, "distance": rayon_km, "size": 20})
    if not stations:
        return {"disponible": False, "raison": f"aucune station à moins de {rayon_km} km"}

    # Trier par distance, garder la première station qui mesure un débit
    stations.sort(key=lambda s: distance_km(lat, lon, s["latitude_station"], s["longitude_station"]))
    for s in stations:
        code = s["code_station"]
        obs = hubeau("observations_tr", {"code_entite": code, "grandeur_hydro": "Q",
                                         "size": 1, "sort": "desc"})
        if not obs:
            continue
        debit = obs[0]["resultat_obs"] / 1000   # l/s → m³/s

        # Référence : débits moyens journaliers des 30 derniers jours
        elab = hubeau("obs_elab", {"code_entite": code, "grandeur_hydro_elab": "QmnJ",
                                   "date_debut_obs_elab": str(date.today() - timedelta(days=30)),
                                   "size": 40}) or []
        valeurs = sorted(e["resultat_obs_elab"] / 1000 for e in elab if e.get("resultat_obs_elab"))
        mediane = valeurs[len(valeurs)//2] if valeurs else None

        return {
            "disponible": True,
            "station": s.get("libelle_station"),
            "code_station": code,
            "distance_km": round(distance_km(lat, lon, s["latitude_station"], s["longitude_station"]), 1),
            "debit_m3s": round(debit, 2),
            "debit_median_30j_m3s": round(mediane, 2) if mediane else None,
            "ratio": round(debit / mediane, 2) if mediane else None,
        }
    return {"disponible": False, "raison": "stations proches sans mesure de débit"}

print(hubeau_contexte(48.8566, 2.3522))

{'disponible': True, 'station': 'La Seine à Paris - Austerlitz [>2006] - Station débitmétrique', 'code_station': 'F700000103', 'distance_km': 1.6, 'debit_m3s': 76.7, 'debit_median_30j_m3s': 81.04, 'ratio': 0.95}


In [15]:
def appliquer_debit(r, hub):
    """Ajoute les règles Hub'Eau à un résultat de evaluer_risque."""
    h, a = r["risque_humains"]["score"], r["risque_animaux"]["score"]
    ratio = hub.get("ratio") if hub.get("disponible") else None

    if ratio is not None:
        if ratio >= 2:
            h += 2; a += 1
            r["raisons"].append(f"débit {ratio} fois supérieur à la normale : crue, courant dangereux")
        elif ratio >= 1.5:
            h += 1
            r["raisons"].append(f"débit élevé ({ratio} fois la normale)")
        elif ratio <= 0.3:
            a += 1
            r["raisons"].append(f"débit très faible ({ratio} fois la normale) : pollution plus concentrée")

    nh, na = niveau(h), niveau(a)
    r["risque_humains"] = {"niveau": nh, "score": h, "consigne": CONSIGNES["humains"][nh]}
    r["risque_animaux"] = {"niveau": na, "score": a, "consigne": CONSIGNES["animaux"][na]}
    r["a_valider_gestionnaire"] = "ELEVE" in (nh, na)
    r["hubeau"] = hub
    if r["raisons"][0] == "aucun signal de risque détecté" and len(r["raisons"]) > 1:
        r["raisons"].pop(0)
    return r

def evaluer_risque_complet(fiche, lat, lon, meteo=None, hub=None):
    meteo = meteo or meteo_contexte(lat, lon)
    hub = hub or hubeau_contexte(lat, lon)
    return appliquer_debit(evaluer_risque(fiche, meteo), hub)

# --- Test réel (Paris) + test de crue simulée ---
meteo = meteo_contexte(48.8566, 2.3522)
hub = hubeau_contexte(48.8566, 2.3522)          # appelé une seule fois pour toutes les fiches
hub_crue = dict(hub, debit_m3s=250, ratio=3.1)  # scénario de démo

resultats_j5 = []
for f in fiches:
    r = evaluer_risque_complet(f, 48.8566, 2.3522, meteo, hub)
    r["photo"] = f.get("photo")
    resultats_j5.append(r)
    rc = evaluer_risque_complet(f, 48.8566, 2.3522, meteo, hub_crue)
    print(f"{str(f.get('photo'))[-26:]:26} | réel : H {r['risque_humains']['niveau']:6} A {r['risque_animaux']['niveau']:6} | crue : H {rc['risque_humains']['niveau']:6} A {rc['risque_animaux']['niveau']:6}")

with open('/content/drive/MyDrive/streamsentinel/resultats/resultats_j5.json', 'w') as out:
    json.dump(resultats_j5, out, indent=2, ensure_ascii=False)

berge_betonnee_00.jpg      | réel : H FAIBLE A FAIBLE | crue : H MODERE A FAIBLE
berge_betonnee_01.jpg      | réel : H FAIBLE A FAIBLE | crue : H MODERE A FAIBLE
berge_naturelle_00.jpg     | réel : H FAIBLE A FAIBLE | crue : H MODERE A FAIBLE
berge_naturelle_01.jpg     | réel : H FAIBLE A FAIBLE | crue : H MODERE A FAIBLE
eau_boueuse_00.jpg         | réel : H FAIBLE A FAIBLE | crue : H MODERE A FAIBLE
eau_boueuse_01.jpg         | réel : H FAIBLE A FAIBLE | crue : H MODERE A MODERE
eau_verte_02.jpg           | réel : H MODERE A MODERE | crue : H ELEVE  A MODERE
mousse_00.jpg              | réel : H FAIBLE A FAIBLE | crue : H MODERE A FAIBLE
mousse_01.jpg              | réel : H MODERE A MODERE | crue : H ELEVE  A ELEVE 
mousse_02.jpg              | réel : H FAIBLE A FAIBLE | crue : H MODERE A FAIBLE
dechets_000.jpg            | réel : H ELEVE  A ELEVE  | crue : H ELEVE  A ELEVE 
dechets_001.jpg            | réel : H ELEVE  A ELEVE  | crue : H ELEVE  A ELEVE 
dechets_002.jpg            |

In [16]:
# Exécution du J5 sur les fiches du J4
tout = j4 if isinstance(j4, list) else list(j4.values())
fiches = [f for f in tout if "formulaire" in f]
print(f"{len(fiches)} fiches exploitables, {len(tout) - len(fiches)} photos refusées ignorées")

LAT, LON = 48.8566, 2.3522   # coordonnées de test : Seine à Paris
meteo = meteo_contexte(LAT, LON)
hub = hubeau_contexte(LAT, LON)
print("Météo :", meteo)
print("Hub'Eau :", hub)

resultats_j5 = []
for f in fiches:
    r = evaluer_risque_complet(f, LAT, LON, meteo, hub)
    r["photo"] = f.get("photo")
    resultats_j5.append(r)
    print(f"{str(f.get('photo'))[-26:]:26} | humains {r['risque_humains']['niveau']:6} | animaux {r['risque_animaux']['niveau']:6}")

with open('/content/drive/MyDrive/streamsentinel/resultats/resultats_j5.json', 'w') as out:
    json.dump(resultats_j5, out, indent=2, ensure_ascii=False)

15 fiches exploitables, 4 photos refusées ignorées
Météo : {'disponible': True, 'pluie_72h_mm': 0, 'pluie_aujourdhui_mm': 0, 'pluie_prevue_3j_mm': 0, 'temp_max_aujourdhui': 24.5, 'temp_max_3j': 25.5}
Hub'Eau : {'disponible': True, 'station': 'La Seine à Paris - Austerlitz [>2006] - Station débitmétrique', 'code_station': 'F700000103', 'distance_km': 1.6, 'debit_m3s': 76.7, 'debit_median_30j_m3s': 81.04, 'ratio': 0.95}
berge_betonnee_00.jpg      | humains FAIBLE | animaux FAIBLE
berge_betonnee_01.jpg      | humains FAIBLE | animaux FAIBLE
berge_naturelle_00.jpg     | humains FAIBLE | animaux FAIBLE
berge_naturelle_01.jpg     | humains FAIBLE | animaux FAIBLE
eau_boueuse_00.jpg         | humains FAIBLE | animaux FAIBLE
eau_boueuse_01.jpg         | humains FAIBLE | animaux FAIBLE
eau_verte_02.jpg           | humains MODERE | animaux MODERE
mousse_00.jpg              | humains FAIBLE | animaux FAIBLE
mousse_01.jpg              | humains MODERE | animaux MODERE
mousse_02.jpg              | 